### Import Dependencies

In [4]:
from google import genai
from google.genai import types
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.http.models import  VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Retrieval Pipeline

In [5]:
query = "best laptop for programming"

In [6]:
import os
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

In [7]:
def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="gemini-embedding-001",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [8]:
results = retrieve_data(query, k=20)

In [9]:
results

{'retrieved_context_ids': ['B09J52P1TD',
  'B09XWK1KV7',
  'B0C9ZWCZ99',
  'B0BWD5TKNH',
  'B0B9JGHGDW',
  'B09YRY6QCX',
  'B0B9T5KSRN',
  'B08TWJR3N3',
  'B0BSNHN56H',
  'B09XJJNCXZ',
  'B0CF5CQ48T',
  'B09ZHCRSJL',
  'B0CC3KFS16',
  'B0B9XSS7DC',
  'B0BHSX2P7S',
  'B09XBFN4NR',
  'B0B5HVSQ2L',
  'B0B1HN1DK2',
  'B0B51SZ6J9',
  'B0C8BK3TNY'],
 'retrieved_context': ['HP 17.3" Flagship HD+ Business Laptop, Intel Quad Core i3-1125G4(Beat i5-1035G4), 8GB RAM, 256GB PCIe SSD, Bluetooth, HDMI, Webcam, Windows 11, Silver, w/GM Accessories 【11th Gen Intel Core i3-1125G4 】2.0 GHz base frequency, up to 3.7 GHz with Intel Turbo Boost Technology, The 11th gen dual-core laptop brings the perfect combination of features to make you unstoppable. This is an ideal home office laptop to get things done fast with high performance, instant responsiveness and best-in-class connectivity. 【17.3" HD+(1600x900) Anti-Glare LED IPS Non-Touch Display】17.3-inch diagonal, HD+, BrightView, 220 nits, 60% NTSC，boasts

### Reranking

In [11]:
cohere_client = cohere.ClientV2(os.getenv("CO_API_KEY"))

In [12]:
to_rerank = results["retrieved_context"]

In [14]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20,
)

In [15]:
response

V2RerankResponse(id='3430926d-4723-4d6b-8390-8358b7ea055c', results=[V2RerankResponseResultsItem(index=4, relevance_score=0.7022034), V2RerankResponseResultsItem(index=2, relevance_score=0.680539), V2RerankResponseResultsItem(index=0, relevance_score=0.6312306), V2RerankResponseResultsItem(index=9, relevance_score=0.5521533), V2RerankResponseResultsItem(index=16, relevance_score=0.5171807), V2RerankResponseResultsItem(index=19, relevance_score=0.49375027), V2RerankResponseResultsItem(index=5, relevance_score=0.47424152), V2RerankResponseResultsItem(index=3, relevance_score=0.4664567), V2RerankResponseResultsItem(index=13, relevance_score=0.45868814), V2RerankResponseResultsItem(index=11, relevance_score=0.45093957), V2RerankResponseResultsItem(index=18, relevance_score=0.43936244), V2RerankResponseResultsItem(index=1, relevance_score=0.4355173), V2RerankResponseResultsItem(index=12, relevance_score=0.40507212), V2RerankResponseResultsItem(index=17, relevance_score=0.40131232), V2Rerank

In [16]:
rerank_results = [to_rerank[result.index] for result in response.results]

In [17]:
rerank_results

['ASUS ZenBook 14X OLED Laptop, 14” WQXGA+ 16:10 Touch Display, Intel Core i7-1165G7 CPU, NVIDIA GeForce MX450, 16GB RAM, 512GB SSD, Windows 11 Pro, Pine Grey, UX5400EG-XB73T Aspect Ratio:16:10 Complimentary 3-month Adobe Creative Cloud subscription with the purchase. Learn more on ASUS website for more details Innovative ScreenPad: 5.65-inch interactive touchscreen trackpad that adapts to your needs for smarter control and multitasking 14” WQXGA+ (2880 x 1800) 16:10 OLED Touch Screen 400 nits display with ultra-slim 4-sided NanoEdge bezels Latest 11th generation Intel Core i7-1165G7 Quad Core Processor (12M Cache) Discrete NVIDIA GeForce MX450 graphics Fast storage and memory featuring 512GB PCIe NVMe M.2 SSD and 16GB LPDDR4X RAM, with Windows 11 Pro Pantone Validated, DCI-P3: 100% Glossy display with 92% screen-to-body ratio Extensive connectivity with Thunderbolt 4 via USB-C, USB 3.2 Type A, HDMI, 3.5mm Combo Audio Jack, Micro SD card reader, Wi-Fi 6 (802.11ax), and Bluetooth 5.0 Sl